# LNS-Optimierungstest: Heuristische Verbesserung durch Ruin-and-Recreate

Dieses Notebook evaluiert die Leistung und Effektivität des **Large Neighborhood Search (LNS)** Verfahrens (Ruin-and-Recreate), welches in `optimize_multiple` implementiert ist. 

Das LNS-Verfahren versucht, die gierige (greedy) Zuweisung des Dijkstra/A*-Routers nachträglich zu optimieren, indem in jeder Iteration ein Teil der Sendungen entfernt (ruined) und unter Berücksichtigung der verbleibenden Kapazitäten neu geroutet (recreated) wird. Dies führt zu einer besseren Bündelung (Konsolidierung) der Sendungen und senkt somit die Fixkosten und CO₂-Emissionen der aktivierten Fahrzeuge.

Folgende Experimente werden durchgeführt:
1. **Konvergenzanalyse (LNS-Verlauf):** Zielfunktionswert über die Iterationen hinweg (0 bis 50 Iterationen).
2. **Sensitivität der Ruin-Fraction:** Wie beeinflusst der Anteil der temporär gelöschten Sendungen (10%, 20%, 30%, 40%) das Endergebnis und die Laufzeit?
3. **Direkter Vergleich (Greedy vs. LNS):** Gegenüberstellung von Gesamtkosten, CO₂-Emissionen und Konsolidierungsraten.

## 1. Setup und Imports

In [ ]:
import sys
import time
import random
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from freight_routing.data_loader import NetworkDataLoader
from freight_routing.data_models import Shipment, ObjectiveWeights, ArcType
from freight_routing.model import TimeExpandedNetwork
from heuristics.dijkstra_router import AStarRouter

sns.set_theme(style="whitegrid", palette="muted")

## 2. Laden der Netzwerkdaten und Generierung der Test-Sendungen

Wir verwenden eine Testinstanz mit **50 Sendungen** auf dem großen Netzwerk (`large_network.json`). Um Konsolidierungseffekte zu forcieren, wählen wir Sendungen mit mittlerem Gewicht (z.B. 5t bis 25t) aus europäischen Hubs, die sich zeitlich und räumlich überschneiden.

In [ ]:
network_path = PROJECT_ROOT / "dataset/large_network.json"
network_data = NetworkDataLoader.from_json(network_path)
rail_hubs = [h_id for h_id, hub in network_data.hubs.items() if "rail" in hub.supported_modes]

random.seed(42)
N_SHIPMENTS = 50
PLANNING_DAYS = 30
DEADLINE = PLANNING_DAYS * 24 * 60

shipments = []
for i in range(N_SHIPMENTS):
    start = random.choice(rail_hubs)
    dest = random.choice(rail_hubs)
    while dest == start:
        dest = random.choice(rail_hubs)
        
    shipments.append(
        Shipment(
            id=f"shipment_{i}",
            start_hub=start,
            end_hub=dest,
            start_time=360, # Startet um 6:00 Uhr morgens
            deadline=DEADLINE,
            max_price=1000000.0,
            max_emissions=None,
            weight=float(random.randint(5, 25)), # 5 bis 25 Tonnen
            objective_weights=ObjectiveWeights(cost=0.4, time=0.2, emissions=0.4)
        )
    )

network = TimeExpandedNetwork.build(network_data, planning_days=PLANNING_DAYS, shipments=shipments)
print(f"{len(shipments)} Sendungen generiert. Netzwerk erfolgreich gebaut.")

## 3. Hilfsfunktion zur Analyse der Konsolidierungsrate

Diese Funktion ermittelt den Anteil der erfolgreich gelösten Sendungen, die sich mindestens einen Transportbogen mit einer anderen Sendung teilen.

In [ ]:
def get_consolidation_rate(result):
    if result.status == "Infeasible" or not result.shipment_routes:
        return 0.0
        
    # Zähle Sendungen pro Transportbogen
    arc_shipments = defaultdict(list)
    for s_id, route in result.shipment_routes.items():
        for arc in route:
            if arc.arc_type == ArcType.TRANSPORT:
                arc_shipments[arc].append(s_id)
                
    consolidated_shipments = set()
    for arc, s_ids in arc_shipments.items():
        if len(s_ids) >= 2:
            for s_id in s_ids:
                consolidated_shipments.add(s_id)
                
    solved_count = len(result.shipment_routes)
    return (len(consolidated_shipments) / solved_count) * 100 if solved_count > 0 else 0.0

## 4. Experiment 1: Konvergenzverlauf der LNS-Optimierung (Zielfunktionswert vs. Iterationen)

Wir vergleichen die gierige Zuweisung (0 Iterationen) mit LNS-Laufzeiten von 5, 10, 20, 30 und 50 Iterationen bei einer festen Ruin-Fraction von 20%.

In [ ]:
weights_default = ObjectiveWeights(cost=0.4, time=0.2, emissions=0.4)
router = AStarRouter(objective_weights=weights_default)

# 1. Greedy-Ausgangslösung berechnen
print("Starte Greedy-Zuweisung...")
t0 = time.time()
greedy_result = router.solve_multiple(network)
t_greedy = time.time() - t0
print(f"Greedy abgeschlossen in {t_greedy:.2f} s. Zielfunktionswert: {greedy_result.objective_value:.6f}")

# 2. LNS mit steigenden Iterationen ausführen
iterations_to_test = [0, 5, 10, 20, 30, 50]
convergence_results = []

for iters in iterations_to_test:
    if iters == 0:
        res = greedy_result
        dur = 0.0
    else:
        print(f"Starte LNS mit {iters} Iterationen...")
        t_start = time.time()
        res = router.optimize_multiple(
            initial_result=greedy_result,
            network=network,
            iterations=iters,
            ruin_fraction=0.2,
            seed=42
        )
        dur = time.time() - t_start
        print(f"LNS ({iters} iters) abgeschlossen in {dur:.2f} s. Zielfunktionswert: {res.objective_value:.6f}")
        
    c_rate = get_consolidation_rate(res)
    convergence_results.append({
        "iterations": iters,
        "objective_value": res.objective_value,
        "total_cost": res.total_cost,
        "total_emissions": res.total_emissions,
        "consolidation_rate": c_rate,
        "duration_lns": dur
    })

df_conv = pd.DataFrame(convergence_results)

### Visualisierung von Experiment 1: Konvergenzverlauf

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Zielfunktionswert (Minderung = Verbesserung)
sns.lineplot(
    data=df_conv, x="iterations", y="objective_value",
    marker="o", color="b", linewidth=2.5, ax=axes[0]
)
axes[0].set_title("LNS-Konvergenz: Zielfunktionswert vs. Iterationen", fontsize=13, fontweight="bold")
axes[0].set_xlabel("LNS-Iterationen", fontsize=11)
axes[0].set_ylabel("Kombinierter Zielfunktionswert", fontsize=11)

# Plot 2: Konsolidierungsrate
sns.lineplot(
    data=df_conv, x="iterations", y="consolidation_rate",
    marker="s", color="g", linewidth=2.5, ax=axes[1]
)
axes[1].set_title("LNS-Bündelungseffekt: Konsolidierungsrate vs. Iterationen", fontsize=13, fontweight="bold")
axes[1].set_xlabel("LNS-Iterationen", fontsize=11)
axes[1].set_ylabel("Konsolidierungsrate (%)", fontsize=11)
axes[1].set_ylim(-5, 105)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "dataset/lns_convergence_plots.png", dpi=300)
plt.show()

## 5. Experiment 2: Sensitivitätsanalyse der Ruin-Fraction

Die Ruin-Fraction steuert, wie viel Prozent der Sendungen pro LNS-Schritt zerstört und neu verplant werden. Wir testen Werte von 10%, 20%, 30% und 40% bei jeweils festen 20 Iterationen.

In [ ]:
ruin_fractions = [0.1, 0.2, 0.3, 0.4]
ruin_results = []

for rf in ruin_fractions:
    print(f"Starte LNS mit ruin_fraction={rf:.1f}...")
    t_start = time.time()
    res = router.optimize_multiple(
        initial_result=greedy_result,
        network=network,
        iterations=20,
        ruin_fraction=rf,
        seed=42
    )
    dur = time.time() - t_start
    c_rate = get_consolidation_rate(res)
    
    ruin_results.append({
        "ruin_fraction": rf * 100,
        "objective_value": res.objective_value,
        "total_cost": res.total_cost,
        "total_emissions": res.total_emissions,
        "consolidation_rate": c_rate,
        "duration": dur
    })
    print(f"Ruin fraction {rf:.1f} abgeschlossen in {dur:.2f} s. Zielfunktionswert: {res.objective_value:.6f}")

df_ruin = pd.DataFrame(ruin_results)

### Visualisierung von Experiment 2: Sensitivität der Ruin-Fraction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Zielfunktionswert vs. Ruin-Fraction
sns.barplot(
    data=df_ruin, x="ruin_fraction", y="objective_value",
    color="coral", ax=axes[0]
)
# Zeichne eine rote Linie für das Greedy-Ausgangsniveau (0 Iterationen / 0% Ruin)
axes[0].axhline(greedy_result.objective_value, color="red", linestyle="--", linewidth=2, label="Greedy (Ausgangswert)")
axes[0].set_title("Lösungsqualität in Abhängigkeit von der Ruin-Fraction", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Ruin-Fraction (% der Sendungen)", fontsize=11)
axes[0].set_ylabel("Zielfunktionswert (niedriger ist besser)", fontsize=11)
axes[0].legend()

# Plot 2: Rechenzeit vs. Ruin-Fraction
sns.lineplot(
    data=df_ruin, x="ruin_fraction", y="duration",
    marker="o", color="purple", linewidth=2.5, ax=axes[1]
)
axes[1].set_title("Rechenzeit in Abhängigkeit von der Ruin-Fraction", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Ruin-Fraction (% der Sendungen)", fontsize=11)
axes[1].set_ylabel("LNS-Rechenzeit (s)", fontsize=11)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "dataset/lns_ruin_plots.png", dpi=300)
plt.show()

## 6. Zusammenfassender Vergleich: Greedy vs. LNS-Optimierung

Wir vergleichen die absolute und relative Verbesserung der Zielmetriken zwischen der gierigen Ausgangslösung und der durch 50 LNS-Iterationen (ruin_fraction = 20%) optimierten Endlösung.

In [ ]:
best_lns = df_conv.loc[df_conv["iterations"] == 50].iloc[0]
greedy_val = df_conv.loc[df_conv["iterations"] == 0].iloc[0]

improvement_obj = ((greedy_val["objective_value"] - best_lns["objective_value"]) / greedy_val["objective_value"]) * 100
improvement_cost = ((greedy_val["total_cost"] - best_lns["total_cost"]) / greedy_val["total_cost"]) * 100
improvement_emissions = ((greedy_val["total_emissions"] - best_lns["total_emissions"]) / greedy_val["total_emissions"]) * 100

print("========================================================================")
print("            Vergleich: Greedy-Heuristik vs. LNS-Optimierung (50 iters)    ")
print("========================================================================")
print(f"Metrik                | Greedy-Ausgang  | LNS-Optimiert   | Verbesserung ")
print("------------------------------------------------------------------------")
print(f"Zielfunktionswert     | {greedy_val['objective_value']:.6f}        | {best_lns['objective_value']:.6f}        | {improvement_obj:.2f} %")
print(f"Gesamtkosten          | {greedy_val['total_cost']:.2f} €    | {best_lns['total_cost']:.2f} €    | {improvement_cost:.2f} %")
print(f"Gesamtemissionen      | {greedy_val['total_emissions']:.2f} kg    | {best_lns['total_emissions']:.2f} kg    | {improvement_emissions:.2f} %")
print(f"Konsolidierungsrate   | {greedy_val['consolidation_rate']:.2f} %       | {best_lns['consolidation_rate']:.2f} %       | + {best_lns['consolidation_rate'] - greedy_val['consolidation_rate']:.2f} %-Punkte")
print("========================================================================")